# AIPerf Use Case 2 — Auditing Raw Per-Request Exports

Every AIPerf run exports **every individual request** to `profile_export.jsonl`. This notebook loads that raw export and computes statistics AIPerf's summary table doesn't ship — e.g. a P75, when the report only has P50/P90/P99.

These use-case notebooks demonstrate AIPerf capabilities directly. This is the foundation of the repo's convention that **the raw AIPerf export is the deliverable** — no processed report layer.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — anatomy of `profile_export.jsonl`](#3-input--anatomy-of-profile_exportjsonl)
4. [Results analysis](#4-results-analysis)

## 1. Overview

The summary table (CSV/JSON) gives avg/min/max/p50/p90/p99/std per metric. The JSONL gives the underlying per-request records, so you can:

- compute **any percentile or custom statistic** (P75, trimmed means, ...);
- **slice** by input/output length, worker, or time window;
- build your own dashboards — the format is open, nothing is locked to AIPerf's presentation.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). Pin whatever version you use (repo convention: record the AIPerf version per run) — this notebook was written against `release/0.3.0`.
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.
- **Data**: an existing AIPerf artifact directory. Point `ARTIFACT_DIR` (below) at any prior run — e.g. this repo's scenario outputs, or the Use Case 1 notebook's `notebooks/artifacts/aiperf-uc1-synthetic`.

In [ ]:
# Relative to REPO_ROOT — any prior run's --artifact-dir (kept under notebooks/).
ARTIFACT_DIR = "notebooks/artifacts/aiperf-uc1-synthetic"
print(f"Artifact dir: {ARTIFACT_DIR}")

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — anatomy of `profile_export.jsonl`

One JSON object per line, one line per request. Abridged example record:

```json
{
  "metadata": {
    "session_num": 87,
    "x_request_id": "abd8df1a-7904-4aa0-8107-0d74ba0ac0d7",
    "turn_index": 0,
    "request_start_ns": 1763066701865462000,
    "request_end_ns": 1763066703082535666,
    "worker_id": "worker_b431129c"
  },
  "metrics": {
    "time_to_first_token": {"value": 582.66, "unit": "ms"},
    "request_latency": {"value": 1210.008, "unit": "ms"},
    "inter_token_latency": {"value": 3.25, "unit": "ms"},
    "input_sequence_length": {"value": 1000, "unit": "tokens"},
    "output_sequence_length": {"value": 194, "unit": "tokens"}
  }
}
```

Send 10,000 requests and you get 10,000 of these. Field names/nesting can vary slightly across AIPerf versions — the loader below flattens whatever `metrics` contains.

## 4. Results analysis

Load the JSONL into a flat DataFrame (one row per request, one column per metric), then compute custom statistics.

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / ARTIFACT_DIR
jsonl_path = next(artifact_dir.rglob("profile_export.jsonl"), None)
assert jsonl_path is not None, f"No profile_export.jsonl under {artifact_dir}."

records = []
with open(jsonl_path) as f:
    for line in f:
        rec = json.loads(line)
        row = {k: v for k, v in rec.get("metadata", {}).items()}
        for name, m in rec.get("metrics", {}).items():
            row[name] = m["value"] if isinstance(m, dict) and "value" in m else m
        records.append(row)

df = pd.DataFrame(records)
print(f"{len(df)} requests, columns: {sorted(df.columns)}")

# First 5 raw per-request records, flattened to one row per request — metadata
# columns (session/request id, worker, timestamps) plus one column per metric.
df.head()


In [ ]:
# A motivating example: percentiles the summary table doesn't include.
ttft = df["time_to_first_token"].dropna()

percentiles = [25, 50, 75, 90, 99]
custom = pd.DataFrame({
    "percentile": [f"P{p}" for p in percentiles],
    "ttft_ms": [ttft.quantile(p / 100) for p in percentiles],
})
print(f"Total requests analyzed: {len(ttft)}")

# Custom TTFT percentiles (incl. P25/P75) computed directly from the raw
# per-request data — not present in AIPerf's summary table (which only ships P50/P90/P99).
custom


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ttft.plot.hist(bins=40, ax=axes[0])
axes[0].set_title("TTFT distribution")
axes[0].set_xlabel("ms")

axes[1].plot(ttft.sort_values().values, ttft.rank(pct=True).sort_values().values)
axes[1].set_title("TTFT ECDF (read off any percentile)")
axes[1].set_xlabel("ms")
axes[1].set_ylabel("fraction of requests")

if {"input_sequence_length", "request_latency"} <= set(df.columns):
    axes[2].scatter(df["input_sequence_length"], df["request_latency"], s=8, alpha=0.4)
    axes[2].set_title("Request latency vs. input length")
    axes[2].set_xlabel("ISL (tokens)")
    axes[2].set_ylabel("latency (ms)")

plt.tight_layout()


### Notes

- Group by `worker_id` to spot client-side load-generator imbalance; use `request_start_ns` to re-derive arrival patterns or your own time buckets (AIPerf can also do this natively — see the Use Case 5 notebook).
- This post-hoc workflow is exactly how this repo computes goodput today (`--goodput` isn't passed in the scenario scripts) — see the Use Case 4 notebook.